In [ ]:
"""
Notebook 05: Pricing Optimization and Scenario Simulation
==========================================================
Implements profit-aware pricing optimization for focus categories using
the Lerner rule. Conducts cost sensitivity analysis across COGS scenarios.
Analyzes sequential purchase patterns and designs recommendation timing
framework based on customer return behavior.

Paper: Profit-Aware Pricing in Two-Sided Marketplaces
Author: Dinesh R. Poddaturi, Ph.D.
SSRN: https://ssrn.com/abstract=6502262

Requires: Run notebook 01 first. Also requires outputs/elasticity_final_summary.csv
from notebook 02.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from collections import Counter
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Price optimization:

# Optimal Price = (MC × eta) / (eta + 1)

# Where:
# - MC = Marginal Cost
# - eta = Price elasticity (negative value)

# Simplified (assuming constant markup):
# Optimal Price ≈ Current Price × (1 + adjustment factor)


In [ ]:
# Loading the elasticity results
# 02_segmentation_elasticity_clv.ipynb generates the elasticity_final_summary.csv file
elasticity_summary = pd.read_csv('../outputs/elasticity_final_summary.csv')

elastcities_negative = elasticity_summary[elasticity_summary['Elasticity_Controlled'] < 0].copy()

# Using the four robust elastic categories
elastic_categories = {
    'watches_gifts': {
        'elasticity': -2.98,
        'current_avg_price': None,  # We'll calculate from data
        'r_squared': 0.89,
        'significance': 'p<0.01'
    },
    'garden_tools': {
        'elasticity': -2.77,
        'current_avg_price': None,
        'r_squared': 0.72,
        'significance': 'p<0.01'
    },
    'electronics': {
        'elasticity': -2.18,
        'current_avg_price': None,
        'r_squared': 0.62,
        'significance': 'p<0.01'
    }
}

# Loading transactions to get current data
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
products_english = pd.read_csv('../data/product_category_name_translation.csv')
products = products.merge(products_english, on = 'product_category_name', how='inner')

# Merge to get categories
transactions = order_items.merge(
    products[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

print(f"Total transactions: {len(transactions):,}")
print(f"Transactions with category: {transactions['product_category_name_english'].notna().sum():,}")


# Calculate current average prices for our elastic categories
for cat_name, cat_data in elastic_categories.items():
    category_transactions = transactions[transactions['product_category_name_english'] == cat_name]

    if len(category_transactions)>0:
        avg_price = category_transactions['price'].mean()
        median_price = category_transactions['price'].median()
        total_revenue = category_transactions['price'].sum()
        num_orders = len(category_transactions)

        elastic_categories[cat_name]['current_avg_price'] = avg_price
        elastic_categories[cat_name]['median_price'] = median_price
        elastic_categories[cat_name]['current_revenue'] = total_revenue
        elastic_categories[cat_name]['num_orders'] = num_orders
        print(f"\n{cat_name.replace('_', ' ').title()}:")
        print(f"Average price: BRL {avg_price:.2f}")
        print(f"Median price: BRL {median_price:.2f}")
        print(f"Total orders: {num_orders:,}")
        print(f"Total revenue: BRL {total_revenue:,.2f}")
        print(f"Elasticity: {cat_data['elasticity']:.2f}")


In [ ]:
# Calculate Optimal Prices:

# Price optimization analysis

print("\n Maximizing profit (not revenue)")
print("  Profit = (Price - Cost) x Quantity")
print("  Revenue = Price x Quantity")
print("\nCost Assumptions:")
print("Low Cost: 60% of current price (40% margin)")
print("Medium Cost: 65% of current price (35% margin)")
print("High Cost: 70% of current price (30% margin)")

optimization_results = []

# Define elasticity scenarios
scenarios = {
    'Conservative': 0.8,   # Elasticity 20% lower
    'Base Case': 1.0,      # Your estimated elasticity
    'Aggressive': 1.2      # Elasticity 20% higher
}

# Define cost scenarios
cost_scenarios = {
    'Low Cost (60%)': 0.60,
    'Medium Cost (65%)': 0.65,
    'High Cost (70%)': 0.70
}


for cat_name, cat_data in elastic_categories.items():
    if cat_data['current_avg_price'] is not None:
        current_price = cat_data['current_avg_price']
        current_orders = cat_data['num_orders']
        current_revenue = cat_data['current_revenue']
        base_elasticity = cat_data['elasticity']
        
        print(f"\n{'='*80}")
        print(f"{cat_name.replace('_', ' ').title().upper()}")
        print(f"{'='*80}")
        print(f"Base elasticity: {base_elasticity:.2f} (R² = {cat_data['r_squared']:.2f})")
        print(f"Current: BRL {current_price:.2f} | {current_orders:,} orders | BRL {current_revenue:,.0f} revenue")
        
        # Focus on Base Case elasticity + Medium Cost for main results
        scenario_elasticity = base_elasticity  # Base case
        cost_pct = 0.65  # Medium cost assumption
        
        unit_cost = current_price * cost_pct
        current_profit = (current_price - unit_cost) * current_orders
        
        print(f"\nCost Structure (Medium 65%):")
        print(f"Unit cost: BRL {unit_cost:.2f}")
        print(f"Margin per unit: BRL {current_price - unit_cost:.2f} ({(1-cost_pct)*100:.0f}%)")
        print(f"Current profit: BRL {current_profit:,.0f}")
        
        # Test price changes from -40% to +40%
        price_changes = np.linspace(-0.40, 0.40, 161)
        
        test_results = []
        for price_change in price_changes:
            new_price = current_price * (1 + price_change)
            
            # Cost stays constant (it's per unit, not % of price)
            # Note: Cost doesn't change when you change price
            new_cost = unit_cost
            
            # Demand response
            demand_multiplier = (1 + price_change) ** scenario_elasticity
            new_orders = current_orders * demand_multiplier
            
            # Revenue and Profit
            new_revenue = new_price * new_orders
            new_profit = (new_price - new_cost) * new_orders
            
            test_results.append({
                'price_change_pct': price_change * 100,
                'new_price': new_price,
                'new_orders': new_orders,
                'new_revenue': new_revenue,
                'new_profit': new_profit,
                'margin_pct': ((new_price - new_cost) / new_price) * 100
            })
        
        test_df = pd.DataFrame(test_results)
        
        # Find optimal price (max profit, not revenue)
        optimal_idx = test_df['new_profit'].idxmax()
        optimal = test_df.iloc[optimal_idx]
        
        # Also find revenue-maximizing price for comparison
        revenue_optimal_idx = test_df['new_revenue'].idxmax()
        revenue_optimal = test_df.iloc[revenue_optimal_idx]
        
        print(f"\n{'─'*80}")
        print("Profit maximizing price:")
        print(f"{'─'*80}")
        print(f"  Price: BRL {optimal['new_price']:.2f} ({optimal['price_change_pct']:+.1f}%)")
        print(f"  Orders: {optimal['new_orders']:,.0f} ({(optimal['new_orders']/current_orders-1)*100:+.1f}%)")
        print(f"  Revenue: BRL {optimal['new_revenue']:,.0f} ({(optimal['new_revenue']/current_revenue-1)*100:+.1f}%)")
        print(f"  Profit: BRL {optimal['new_profit']:,.0f} ({(optimal['new_profit']/current_profit-1)*100:+.1f}%)")
        print(f"  Margin: {optimal['margin_pct']:.1f}%")
        
        print(f"\n{'─'*80}")
        print("Revenue maximizing price:")
        print(f"{'─'*80}")
        print(f"  Price: BRL {revenue_optimal['new_price']:.2f} ({revenue_optimal['price_change_pct']:+.1f}%)")
        print(f"  Revenue: BRL {revenue_optimal['new_revenue']:,.0f} ({(revenue_optimal['new_revenue']/current_revenue-1)*100:+.1f}%)")
        print(f"  Profit: BRL {revenue_optimal['new_profit']:,.0f} ({(revenue_optimal['new_profit']/current_profit-1)*100:+.1f}%)")
        print(f"  Profit is {((optimal['new_profit']/revenue_optimal['new_profit'])-1)*100:.1f}% higher at profit-optimal price!")
        
        # Store base case results
        optimization_results.append({
            'category': cat_name,
            'elasticity_scenario': 'Base Case',
            'cost_scenario': 'Medium (65%)',
            'elasticity': scenario_elasticity,
            'cost_pct': cost_pct,
            'current_price': current_price,
            'optimal_price': optimal['new_price'],
            'price_change_pct': optimal['price_change_pct'],
            'current_orders': current_orders,
            'optimal_orders': optimal['new_orders'],
            'current_revenue': current_revenue,
            'optimal_revenue': optimal['new_revenue'],
            'current_profit': current_profit,
            'optimal_profit': optimal['new_profit'],
            'profit_gain': optimal['new_profit'] - current_profit,
            'profit_gain_pct': (optimal['new_profit']/current_profit-1)*100,
            'optimal_margin_pct': optimal['margin_pct']
        })

opt_results_df = pd.DataFrame(optimization_results)

# ============================================================================
# SUMMARY: BASE CASE RESULTS
# ============================================================================

print("\n" + "="*80)
print("Total Impact - Base Case (Medium Cost 65%)")
print("="*80)

total_current_profit = opt_results_df['current_profit'].sum()
total_optimal_profit = opt_results_df['optimal_profit'].sum()
total_profit_gain = total_optimal_profit - total_current_profit

print(f"\nCurrent total profit:  BRL {total_current_profit:,.0f}")
print(f"Optimal total profit:  BRL {total_optimal_profit:,.0f}")
print(f"Profit gain:           BRL {total_profit_gain:+,.0f} ({(total_profit_gain/total_current_profit)*100:+.1f}%)")

print("\n" + "─"*80)
print("By Category:")
print("─"*80)
for idx, row in opt_results_df.iterrows():
    print(f"\n{row['category'].replace('_', ' ').title()}:")
    print(f"  Price change: {row['price_change_pct']:+.1f}%")
    print(f"  Profit gain: BRL {row['profit_gain']:+,.0f} ({row['profit_gain_pct']:+.1f}%)")

# Save base case results
opt_results_df.to_csv('../outputs/price_optimization_base_case.csv', index=False)

# ============================================================================
# FULL SENSITIVITY ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("Full Sensitivity Analysis")
print("="*80)
print("Testing all combinations of elasticity x cost scenarios\n")

full_sensitivity_results = []

for cat_name, cat_data in elastic_categories.items():
    if cat_data['current_avg_price'] is not None:
        current_price = cat_data['current_avg_price']
        current_orders = cat_data['num_orders']
        current_revenue = cat_data['current_revenue']
        base_elasticity = cat_data['elasticity']
        
        for elasticity_scenario_name, elasticity_multiplier in scenarios.items():
            scenario_elasticity = base_elasticity * elasticity_multiplier
            
            for cost_scenario_name, cost_pct in cost_scenarios.items():
                unit_cost = current_price * cost_pct
                current_profit = (current_price - unit_cost) * current_orders
                
                # Optimize for this scenario
                price_changes = np.linspace(-0.40, 0.40, 161)
                test_results = []
                
                for price_change in price_changes:
                    new_price = current_price * (1 + price_change)
                    new_cost = unit_cost
                    demand_multiplier = (1 + price_change) ** scenario_elasticity
                    new_orders = current_orders * demand_multiplier
                    new_profit = (new_price - new_cost) * new_orders
                    
                    test_results.append({
                        'price_change_pct': price_change * 100,
                        'new_price': new_price,
                        'new_profit': new_profit
                    })
                
                test_df = pd.DataFrame(test_results)
                optimal_idx = test_df['new_profit'].idxmax()
                optimal = test_df.iloc[optimal_idx]
                
                full_sensitivity_results.append({
                    'category': cat_name,
                    'elasticity_scenario': elasticity_scenario_name,
                    'cost_scenario': cost_scenario_name,
                    'elasticity': scenario_elasticity,
                    'cost_pct': cost_pct,
                    'optimal_price': optimal['new_price'],
                    'price_change_pct': optimal['price_change_pct'],
                    'current_profit': current_profit,
                    'optimal_profit': optimal['new_profit'],
                    'profit_gain': optimal['new_profit'] - current_profit,
                    'profit_gain_pct': (optimal['new_profit']/current_profit-1)*100
                })

full_sens_df = pd.DataFrame(full_sensitivity_results)

# Show sensitivity summary
print("Profit Gain Range Across All Scenarios:")
print("─"*80)

for cat_name in elastic_categories.keys():
    cat_results = full_sens_df[full_sens_df['category'] == cat_name]
    min_gain = cat_results['profit_gain_pct'].min()
    max_gain = cat_results['profit_gain_pct'].max()
    median_gain = cat_results['profit_gain_pct'].median()
    
    print(f"\n{cat_name.replace('_', ' ').title()}:")
    print(f"  Profit gain range: {min_gain:+.1f}% to {max_gain:+.1f}%")
    print(f"  Median gain: {median_gain:+.1f}%")

# Total profit gain by scenario combination
print("\n" + "="*80)
print("Total Profit Gain by Scenario Combination")
print("="*80)

for elasticity_scenario_name in scenarios.keys():
    print(f"\n{elasticity_scenario_name}:")
    for cost_scenario_name in cost_scenarios.keys():
        scenario_results = full_sens_df[
            (full_sens_df['elasticity_scenario'] == elasticity_scenario_name) &
            (full_sens_df['cost_scenario'] == cost_scenario_name)
        ]
        total_gain = scenario_results['profit_gain'].sum()
        total_current = scenario_results['current_profit'].sum()
        
        print(f"  {cost_scenario_name:20s}: BRL {total_gain:10,.0f} ({(total_gain/total_current)*100:+6.1f}%)")

# Save full sensitivity results
full_sens_df.to_csv('../outputs/price_optimization_sensitivity.csv', index=False)

In [ ]:
# Outcomes:

# From the above outputs we found that the revenue maximizing prices are not advised as they would destroy profits.

# We recommend the profit maximizing prices and they gain some profits. Following is the profit optimal recommendations:

# WATCHES GIFTS
# Current Price : 201, optimal price : 197, price change : -2%, revenue gain 4.1% and profit gain 0.1%
# GARDEN TOOLS
# Current Price : 112, optimal price : 113, price change : -1.5%, revenue gain -2.6% and profit gain 0.1%
# ELECTRONICS
# Current Price : 58, optimal price : 70, price change : 20.0%, revenue gain -19.4% and profit gain 5.6%
# CONSOLES GAMES
# Current Price : 138, optimal price : 194, price change : 20.0%, revenue gain -11.1% and profit gain 36.1%

# Total profit gain: 24K (+3.4%)

# Category Strategies:
# 1. Watches/Gifts & Garden Tools: Already Near-Optimal
# Current prices are very close to profit-maximizing
# Only tiny adjustments needed (-2% to +1.5%)
# Insight: Sellers in these categories have already optimized!
# Action: Maintain current pricing, focus on other levers

# 2. Electronics: Moderate Price INCREASE
# Should increase price by +20% (BRL 58 to BRL 70)
# Why? Currently underpriced relative to cost structure
# Elastic demand (eta = -2.18) but low margins (35%)
# Trade-off: Lose 33% of orders, but gain 5.6% profit
# Action: Test gradual price increases

# Phase 1: Quick Wins (Consoles/Games)
# Action: Price increase +40% (BRL 138 to BRL 194)
# Rationale:
# Lowest elasticity (eta = -1.35) to customers less price-sensitive
# Highest profit gain (+36%)
# Robust across all scenarios (+16% to +62%)
# Implementation:
# A/B test +20% first, then +40% if successful
# Position as "premium gaming experience"
# Bundle with accessories to justify higher price
# Expected: +BRL 20K profit (+36%)

# Phase 2: Electronics Price Correction
# Action: Price increase +20% (BRL 58 to BRL 70)
# Rationale:
# Currently underpriced (35% margin too thin)
# Elastic demand but low base price to room to increase
# +5.6% profit gain
# Implementation:
# Gradual rollout (+5%, +10%, +15%, +20%)
# Test on new products first
# Monitor competitor response
# Expected: +BRL 3K profit (+5.6%)

# Phase 3: Maintain Watches & Garden Tools
# Action: No major price changes
# Rationale:
# Already near profit-optimal
# Gains are marginal (0.1%)
# Risk disrupting working pricing strategy
# Focus instead on:
# Volume growth (not price optimization)
# Cross-sell and bundles
# Customer retention programs

In [ ]:
# Creating Visualizations

categories = ['Watches/Gifts', 'Garden Tools', 'Electronics']

# Revenue-Max vs Profit-Max

fig, ax = plt.subplots(figsize = (12, 7))

current_prices = [201.14, 111.63, 57.91]
profit_max_prices = [197.11, 113.30, 69.50]
revenue_max_prices = [120.68, 66.98, 34.75]

x = np.arange(len(categories))
width = 0.25

bars1 = ax.bar(x - width, current_prices, width, label='Current Price',
               color='#95A3A6', alpha=0.8, edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x, profit_max_prices, width, label='Profit-Optimal',
               color='#06A77D', alpha=0.9, edgecolor='black', linewidth=1.5)
bars3 = ax.bar(x + width, revenue_max_prices, width, label='Revenue-Max (Loss)',
               color='#D7263D', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Category', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (BRL)', fontsize=13, fontweight='bold')
ax.set_title('Revenue Maximization vs Profit Maximization\n(Revenue-max causes massive losses)',
             fontsize=15, fontweight='bold', pad=20)

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=12, loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/images/revenue_vs_profit.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Optimal Price Changes
fig, ax = plt.subplots(figsize=(10, 6))

price_changes = [-2.0, 1.5, 20.0]
colors = ['#D7263D' if x < 0 else '#06A77D' for x in price_changes]

bars = ax.bar(categories, price_changes, color=colors, alpha=0.85,
              edgecolor='black', linewidth=1.5)

ax.axhline(0, color='black', linewidth=2, linestyle='-')
ax.set_ylabel('Recommended Price Change (%)', fontsize=13, fontweight='bold')
ax.set_title('Optimal Price Adjustments (Profit-Based)\nMedium Cost 65%',
             fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, price_changes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + (2 if height > 0 else -3),
            f'{val:+.1f}%', ha='center', va='bottom' if height > 0 else 'top',
            fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/images/price_changes.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Profit Gain by Category

fig, ax = plt.subplots(figsize=(10, 6))

profit_gains = [575, 123, 3143]
profit_gain_pcts = [0.1, 0.1, 5.6]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

bars = ax.barh(categories, profit_gains, color=colors, alpha=0.85,
               edgecolor='black', linewidth=1.5)

ax.set_xlabel('Profit Gain (BRL)', fontsize=13, fontweight='bold')
ax.set_title('Profit Gain by Category\nBase Case: Medium Cost 65%',
             fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

for i, (bar, gain, pct) in enumerate(zip(bars, profit_gains, profit_gain_pcts)):
    width = bar.get_width()
    ax.text(width + 500, bar.get_y() + bar.get_height()/2,
            f'BRL {gain:,.0f}\n(+{pct:.1f}%)',
            ha='left', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/images/profit_gains.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


# Sensitivity Analysis Ranges
fig, ax = plt.subplots(figsize=(12, 6))

min_gains = [0.1, 0.0, 0.2]
max_gains = [12.0, 11.9, 29.8]
median_gains = [2.5, 1.8, 5.6]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

y_pos = np.arange(len(categories))

for i, (cat, min_val, max_val, med_val, color) in enumerate(
    zip(categories, min_gains, max_gains, median_gains, colors)):
    
    ax.plot([min_val, max_val], [i, i], 'o-', color=color, 
            linewidth=3, markersize=8, alpha=0.7)
    ax.plot(med_val, i, 'D', color=color, markersize=12,
            markeredgecolor='black', markeredgewidth=2)
    ax.text(max_val + 2, i, f"{min_val:.1f}% to {max_val:.1f}%\n(Median: {med_val:.1f}%)",
            va='center', fontsize=10, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(categories, fontsize=11)
ax.set_xlabel('Profit Gain (%)', fontsize=13, fontweight='bold')
ax.set_title('Sensitivity Analysis: Profit Gain Ranges\nAcross Elasticity (±20%) & Cost Scenarios (60-70%)',
             fontsize=15, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(-5, 50)

plt.tight_layout()
plt.savefig('../outputs/images/sensitivity_ranges.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Scenario Heatmap

fig, ax = plt.subplots(figsize=(10, 7))

elasticity_scenarios_labels = ['Conservative\n(η × 0.8)', 'Base Case\n(η × 1.0)', 'Aggressive\n(η × 1.2)']
cost_scenarios_labels = ['Low Cost\n(60%)', 'Medium Cost\n(65%)', 'High Cost\n(70%)']

data = np.array([
    [35.0, 56.7, 89.6],
    [32.6, 23.7, 35.6],
    [82.0, 34.1, 19.9]
])

im = ax.imshow(data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=90)

ax.set_xticks(np.arange(len(cost_scenarios_labels)))
ax.set_yticks(np.arange(len(elasticity_scenarios_labels)))
ax.set_xticklabels(cost_scenarios_labels, fontsize=11)
ax.set_yticklabels(elasticity_scenarios_labels, fontsize=11)
ax.set_xlabel('Cost Structure', fontsize=13, fontweight='bold')
ax.set_ylabel('Elasticity Scenario', fontsize=13, fontweight='bold')
ax.set_title('Total Profit Gain Across All Scenarios\n(BRL Thousands)',
             fontsize=15, fontweight='bold', pad=20)

for i in range(len(elasticity_scenarios_labels)):
    for j in range(len(cost_scenarios_labels)):
        text = ax.text(j, i, f'BRL {data[i, j]:.0f}K',
                      ha="center", va="center", color="black",
                      fontsize=12, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Profit Gain (BRL K)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/images/scenario_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Current vs Optimal Profit

fig, ax = plt.subplots(figsize=(11, 6))

current_profits = np.array([421752, 169840, 56086]) / 1000
optimal_profits = np.array([422327, 169963, 59229]) / 1000

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, current_profits, width, label='Current Profit',
               color='#95A3A6', alpha=0.8, edgecolor='black', linewidth=1.5)
bars2 = ax.bar(x + width/2, optimal_profits, width, label='Optimal Profit',
               color='#06A77D', alpha=0.9, edgecolor='black', linewidth=1.5)

ax.set_ylabel('Profit (BRL Thousands)', fontsize=13, fontweight='bold')
ax.set_title('Current vs Optimal Profit\nBase Case: Medium Cost 65%',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

for i in range(len(categories)):
    gain_pct = ((optimal_profits[i] / current_profits[i]) - 1) * 100
    if gain_pct > 0.5:
        ax.text(i, max(current_profits[i], optimal_profits[i]) + 10,
                f'+{gain_pct:.1f}%', ha='center', fontsize=10,
                fontweight='bold', color='#06A77D')

plt.tight_layout()
plt.savefig('../outputs/images/profit_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
# Cost sensitivity analysis
# Define categories (excluding Consoles/Games due to weak model fit)
categories_data = {
    'watches_gifts': {
        'name': 'Watches/Gifts',
        'current_price': 201.14,
        'current_orders': 5991,
        'elasticity': -2.98,
        'r_squared': 0.89,
        'typical_cost_range': (0.40, 0.60),  # Watches have better margins
        'color': '#2E86AB'
    },
    'garden_tools': {
        'name': 'Garden Tools',
        'current_price': 111.63,
        'current_orders': 4347,
        'elasticity': -2.77,
        'r_squared': 0.72,
        'typical_cost_range': (0.50, 0.70),  # Moderate margins
        'color': '#A23B72'
    },
    'electronics': {
        'name': 'Electronics',
        'current_price': 57.91,
        'current_orders': 2767,
        'elasticity': -2.18,
        'r_squared': 0.62,
        'typical_cost_range': (0.60, 0.75),  # Thin margins (commodities)
        'color': '#F18F01'
    }
}

# Test cost ratios from 40% to 80%
cost_ratios = np.linspace(0.40, 0.80, 41)

print("Calculating optimal prices across cost scenarios...")

sensitivity_results = []

for cat_key, cat_info in categories_data.items():
    current_price = cat_info['current_price']
    current_orders = cat_info['current_orders']
    elasticity = cat_info['elasticity']
    
    print(f"{cat_info['name']}:")
    print(f"Current price: BRL {current_price:.2f}")
    print(f"Elasticity: {elasticity:.2f} (R-squared = {cat_info['r_squared']:.2f})")
    print(f"Typical cost range: {cat_info['typical_cost_range'][0]*100:.0f}%-{cat_info['typical_cost_range'][1]*100:.0f}%")
    print()
    
    for cost_ratio in cost_ratios:
        unit_cost = current_price * cost_ratio
        current_profit = (current_price - unit_cost) * current_orders
        
        # Test price changes from -50% to +50%
        price_changes = np.linspace(-0.50, 0.50, 201)
        max_profit = -np.inf
        optimal_price = current_price
        optimal_change = 0
        
        for price_change in price_changes:
            new_price = current_price * (1 + price_change)
            demand_multiplier = (1 + price_change) ** elasticity
            new_orders = current_orders * demand_multiplier
            new_profit = (new_price - unit_cost) * new_orders
            
            if new_profit > max_profit:
                max_profit = new_profit
                optimal_price = new_price
                optimal_change = price_change
        
        sensitivity_results.append({
            'category': cat_key,
            'category_name': cat_info['name'],
            'cost_ratio': cost_ratio,
            'cost_pct': cost_ratio * 100,
            'unit_cost': unit_cost,
            'current_price': current_price,
            'optimal_price': optimal_price,
            'optimal_change_pct': optimal_change * 100,
            'current_profit': current_profit,
            'optimal_profit': max_profit,
            'profit_gain': max_profit - current_profit,
            'profit_gain_pct': ((max_profit / current_profit) - 1) * 100 if current_profit > 0 else 0
        })

sens_df = pd.DataFrame(sensitivity_results)

# Save results
sens_df.to_csv('../outputs/cost_sensitivity_analysis.csv', index=False)

print("(Cost ratio where current price is already optimal)\n")

for cat_key, cat_info in categories_data.items():
    cat_data = sens_df[sens_df['category'] == cat_key].copy()
    
    # Find cost ratio closest to 0% price change
    cat_data['abs_change'] = abs(cat_data['optimal_change_pct'])
    breakeven_idx = cat_data['abs_change'].idxmin()
    breakeven = cat_data.loc[breakeven_idx]
    
    print(f"{cat_info['name']}:")
    print(f"Breakeven cost ratio: {breakeven['cost_pct']:.1f}%")
    print(f"Current assumption: 65%")
    print(f"Interpretation: ", end="")
    
    if breakeven['cost_pct'] < 65:
        print(f"If costs < {breakeven['cost_pct']:.0f}%, should INCREASE price")
    elif breakeven['cost_pct'] > 65:
        print(f"If costs > {breakeven['cost_pct']:.0f}%, should DECREASE price")
    else:
        print("Already optimal at 65% cost")
    print()

In [ ]:
# Optimal price changes vs cost ratio
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (cat_key, cat_info) in enumerate(categories_data.items()):
    ax = axes[idx]
    cat_data = sens_df[sens_df['category'] == cat_key]
    
    # Plot optimal price change
    ax.plot(cat_data['cost_pct'], cat_data['optimal_change_pct'],
            color=cat_info['color'], linewidth=3, label='Optimal Price Change')
    
    # Mark current assumption (65%)
    current_assumption = cat_data[cat_data['cost_pct'] == 65].iloc[0]
    ax.plot(65, current_assumption['optimal_change_pct'], 'o', 
            markersize=14, color='red', markeredgecolor='darkred', 
            markeredgewidth=2, label=f"65% Assumption: {current_assumption['optimal_change_pct']:+.0f}%",
            zorder=5)
    
    # Mark typical cost range
    typical_min, typical_max = cat_info['typical_cost_range']
    ax.axvspan(typical_min*100, typical_max*100, alpha=0.15, color=cat_info['color'],
               label=f'Typical Range\n({typical_min*100:.0f}%-{typical_max*100:.0f}%)')
    
    # Zero line
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.4)
    ax.axvline(65, color='red', linewidth=1.5, linestyle='--', alpha=0.4)
    
    ax.set_xlabel('Cost as % of Current Price', fontsize=12, fontweight='bold')
    ax.set_ylabel('Optimal Price Change (%)', fontsize=12, fontweight='bold')
    ax.set_title(f"{cat_info['name']}\n(Elasticity: {cat_info['elasticity']:.2f}, R² = {cat_info['r_squared']:.2f})",
                 fontsize=13, fontweight='bold', pad=10)
    ax.grid(alpha=0.3, linewidth=0.5)
    ax.legend(fontsize=9, loc='best', framealpha=0.9)
    
    # Add annotations with better positioning
    if current_assumption['optimal_change_pct'] > 5:
        direction = "INCREASE"
        color = '#06A77D'
        y_pos = 0.05
    elif current_assumption['optimal_change_pct'] < -5:
        direction = "DECREASE"
        color = '#D7263D'
        y_pos = 0.95
    else:
        direction = "MAINTAIN"
        color = '#95A3A6'
        y_pos = 0.50
    
    if abs(current_assumption['optimal_change_pct']) > 2:
        ax.text(0.05, y_pos, 
                f"At 65% cost:\n{direction} price by\n{abs(current_assumption['optimal_change_pct']):.0f}%",
                transform=ax.transAxes, fontsize=10, fontweight='bold',
                verticalalignment='top' if y_pos > 0.5 else 'bottom',
                bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3, edgecolor='black', linewidth=1))

plt.suptitle('Cost Sensitivity Analysis: How Optimal Price Changes with Cost Assumptions',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../outputs/images/cost_sensitivity_by_category.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# All categories comparison

fig, ax = plt.subplots(figsize=(12, 7))

for cat_key, cat_info in categories_data.items():
    cat_data = sens_df[sens_df['category'] == cat_key]
    ax.plot(cat_data['cost_pct'], cat_data['optimal_change_pct'],
            color=cat_info['color'], linewidth=2.5, label=cat_info['name'],
            marker='o', markersize=3, markevery=5)

# Mark current assumption line
ax.axvline(65, color='red', linewidth=2, linestyle='--', 
           label='Our Assumption (65%)', alpha=0.7)
ax.axhline(0, color='black', linewidth=1.5, linestyle='-', alpha=0.5)

# Shade regions
ax.axhspan(0, 50, alpha=0.1, color='#06A77D', label='Price Increase Zone')
ax.axhspan(-50, 0, alpha=0.1, color='#D7263D', label='Price Decrease Zone')

ax.set_xlabel('Cost as % of Current Price', fontsize=13, fontweight='bold')
ax.set_ylabel('Optimal Price Change (%)', fontsize=13, fontweight='bold')
ax.set_title('Optimal Pricing Strategy vs Cost Structure\n(All Categories)',
             fontsize=15, fontweight='bold', pad=20)
ax.grid(alpha=0.3)
ax.legend(fontsize=11, loc='upper left')
ax.set_xlim(40, 80)
ax.set_ylim(-50, 50)

# Add annotation
ax.text(0.98, 0.02, 
        'Key Insight: Higher costs → Need higher margins → Price increases optimal\n' +
        'Lower costs → Can afford lower margins → Price decreases optimal',
        transform=ax.transAxes, fontsize=10, style='italic',
        verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('../outputs/images/cost_sensitivity_all_categories.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ============================================================================
# Decision Matrix - What to do at different cost levels
# ============================================================================

# Decision matrix

fig, ax = plt.subplots(figsize=(14, 7))

# Create heatmap data
cost_levels = [45, 50, 55, 60, 65, 70, 75, 80]
category_names = [cat_info['name'] for cat_info in categories_data.values()]

heatmap_data = np.zeros((len(category_names), len(cost_levels)))
annotations = []

for i, (cat_key, cat_info) in enumerate(categories_data.items()):
    cat_data = sens_df[sens_df['category'] == cat_key]
    row_annotations = []
    
    for j, cost_level in enumerate(cost_levels):
        # Find closest cost ratio
        closest_idx = (cat_data['cost_pct'] - cost_level).abs().idxmin()
        optimal_change = cat_data.loc[closest_idx, 'optimal_change_pct']
        
        heatmap_data[i, j] = optimal_change
        
        # Create annotation
        if abs(optimal_change) < 2:
            annotation = "Maintain"
        elif optimal_change > 0:
            annotation = f"+{optimal_change:.0f}%"
        else:
            annotation = f"{optimal_change:.0f}%"
        row_annotations.append(annotation)
    
    annotations.append(row_annotations)

# Create heatmap
im = ax.imshow(heatmap_data, cmap='RdYlGn_r', aspect='auto', vmin=-40, vmax=40)

# Set ticks and labels
ax.set_xticks(np.arange(len(cost_levels)))
ax.set_yticks(np.arange(len(category_names)))
ax.set_xticklabels([f'{c}%' for c in cost_levels], fontsize=12, fontweight='bold')
ax.set_yticklabels(category_names, fontsize=12, fontweight='bold')
ax.set_xlabel('Cost as % of Current Price', fontsize=14, fontweight='bold', labelpad=10)
ax.set_ylabel('Category', fontsize=14, fontweight='bold', labelpad=10)

# Fixed title - no overlap
ax.set_title('Optimal Pricing Decision Matrix\n(Recommended price change for each cost scenario)',
             fontsize=16, fontweight='bold', pad=20)

# Add text annotations
for i in range(len(category_names)):
    for j in range(len(cost_levels)):
        text = ax.text(j, i, annotations[i][j],
                      ha="center", va="center", color="black",
                      fontsize=12, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Optimal Price Change (%)', fontsize=13, fontweight='bold', labelpad=10)
cbar.ax.tick_params(labelsize=11)

# Highlight 65% column with BETTER positioning
rect = plt.Rectangle((4.5, -0.5), 1, len(category_names),
                     fill=False, edgecolor='red', linewidth=4)
ax.add_patch(rect)

# Fixed annotation position - moved down to avoid overlap
ax.annotate('Our\nAssumption\n(65%)', 
            xy=(5, -0.5), xytext=(5, -1.2),
            ha='center', fontsize=11, fontweight='bold', color='red',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                     edgecolor='red', linewidth=2))

plt.tight_layout()
plt.savefig('../outputs/images/decision_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ============================================================================
# SUMMARY TABLE: Key Scenarios
# ============================================================================

print("\n" + "="*80)
print("SUMMARY: OPTIMAL STRATEGY BY COST SCENARIO")
print("="*80)

scenarios = [
    ('Low Cost (50%)', 50),
    ('Medium Cost (65%)', 65),
    ('High Cost (75%)', 75)
]

summary_data = []

for scenario_name, cost_level in scenarios:
    print(f"\n{scenario_name}:")
    print("-" * 80)
    
    for cat_key, cat_info in categories_data.items():
        cat_data = sens_df[sens_df['category'] == cat_key]
        closest_idx = (cat_data['cost_pct'] - cost_level).abs().idxmin()
        result = cat_data.loc[closest_idx]
        
        action = "Maintain"
        if result['optimal_change_pct'] > 5:
            action = f"Increase +{result['optimal_change_pct']:.0f}%"
        elif result['optimal_change_pct'] < -5:
            action = f"Decrease {result['optimal_change_pct']:.0f}%"
        
        print(f"  {cat_info['name']:20s}: {action:20s} (Profit: BRL {result['optimal_profit']/1000:.0f}K)")
        
        summary_data.append({
            'Scenario': scenario_name,
            'Category': cat_info['name'],
            'Cost %': cost_level,
            'Optimal Price': f"BRL {result['optimal_price']:.2f}",
            'Price Change': f"{result['optimal_change_pct']:+.1f}%",
            'Action': action,
            'Profit': f"BRL {result['optimal_profit']/1000:.0f}K"
        })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('../outputs/cost_sensitivity_summary.csv', index=False)

In [ ]:
# Sequential purchase analysis is conducted here rather than notebook 02
# because it directly informs the recommendation timing framework
# developed in the optimization section above.

# Key Questions:
# 1. What do customers buy after their first purchase?
# 2. Do customers ever cross categories/buckets over time?
# 3. How long between purchases?
# 4. Can we predict next purchase category?
# 5. How to design loyalty programs to encourage exploration?

# Load datasets
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
product_translation = pd.read_csv('../data/product_category_name_translation.csv')

# Map customer_id to customer_unique_id
# customer_id = order-level (changes per order)
# customer_unique_id = person-level (same person)
customer_mapping = customers[['customer_id', 'customer_unique_id']].drop_duplicates()

print(f"Customer mapping: {len(customer_mapping):,} order-level IDs to {customer_mapping['customer_unique_id'].nunique():,} unique customers")

# Merge to get English categories
products = products.merge(product_translation, on='product_category_name', how='left')

# Create complete transaction dataset
transactions = order_items.merge(
    products[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

transactions = transactions.merge(
    orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

# Add customer_unique_id
transactions = transactions.merge(
    customer_mapping,
    on='customer_id',
    how='left'
)

# Keep only delivered orders
transactions = transactions[transactions['order_status'] == 'delivered'].copy()

# Convert timestamp
transactions['order_date'] = pd.to_datetime(transactions['order_purchase_timestamp'])

print(f"Loaded {len(transactions):,} delivered transactions")
print(f"{transactions['order_id'].nunique():,} unique orders")
print(f"{transactions['customer_unique_id'].nunique():,} unique customers (person-level)")

# Identify repeat customers (using customer_unique_id)
customer_order_counts = transactions.groupby('customer_unique_id')['order_id'].nunique().reset_index()
customer_order_counts.columns = ['customer_unique_id', 'num_orders']

repeat_customers = customer_order_counts[customer_order_counts['num_orders'] > 1]['customer_unique_id'].values
repeat_rate = len(repeat_customers) / len(customer_order_counts) * 100

print(f"\n Repeat Customer Statistics:")
print(f"Total customers: {len(customer_order_counts):,}")
print(f"Repeat customers: {len(repeat_customers):,} ({repeat_rate:.2f}%)")
print(f"One-time buyers: {len(customer_order_counts) - len(repeat_customers):,} ({100-repeat_rate:.2f}%)")

# Distribution of order counts
order_dist = customer_order_counts['num_orders'].value_counts().sort_index()
print(f"\nOrder distribution:")
for num_orders, count in order_dist.head(10).items():
    pct = count / len(customer_order_counts) * 100
    print(f"{num_orders} orders: {count:,} customers ({pct:.2f}%)")


In [ ]:
# Building purchase sequence
# For each order, identify primary category (most items or highest value)
order_categories = transactions.groupby(['order_id', 'product_category_name_english']).agg({
    'price': ['sum', 'count']
}).reset_index()

order_categories.columns = ['order_id', 'category', 'total_price', 'item_count']

# Get primary category (highest spend)
primary_categories = order_categories.loc[
    order_categories.groupby('order_id')['total_price'].idxmax()
][['order_id', 'category']]

# Merge with order info (include customer_unique_id)
order_info = orders[['order_id', 'customer_id', 'order_purchase_timestamp']].merge(
    customer_mapping,
    on='customer_id',
    how='left'
).merge(
    primary_categories,
    on='order_id',
    how='left'
)

order_info['order_date'] = pd.to_datetime(order_info['order_purchase_timestamp'])
order_info = order_info.sort_values(['customer_unique_id', 'order_date'])

# Number orders for each customer (using customer_unique_id)
order_info['order_number'] = order_info.groupby('customer_unique_id').cumcount() + 1

print(f"Created purchase sequences for {order_info['customer_unique_id'].nunique():,} customers")

# Focus on repeat customers
repeat_orders = order_info[order_info['customer_unique_id'].isin(repeat_customers)].copy()

print(f"{len(repeat_orders):,} orders from repeat customers")
print(f"Average orders per repeat customer: {len(repeat_orders)/len(repeat_customers):.2f}")


In [ ]:
# Category transition analysis
# Build transition pairs (order N -> order N+1) using customer_unique_id
transitions = []

for customer_unique_id in repeat_customers:
    customer_orders = repeat_orders[repeat_orders['customer_unique_id'] == customer_unique_id].sort_values('order_date')
    
    for i in range(len(customer_orders) - 1):
        current_order = customer_orders.iloc[i]
        next_order = customer_orders.iloc[i + 1]
        
        time_diff = (next_order['order_date'] - current_order['order_date']).days
        
        transitions.append({
            'customer_unique_id': customer_unique_id,
            'from_category': current_order['category'],
            'to_category': next_order['category'],
            'from_order_num': current_order['order_number'],
            'to_order_num': next_order['order_number'],
            'days_between': time_diff
        })

transitions_df = pd.DataFrame(transitions)

print(f" Identified {len(transitions_df):,} category transitions")
print(f"   Average days between purchases: {transitions_df['days_between'].mean():.1f}")
print(f"   Median days between purchases: {transitions_df['days_between'].median():.1f}")

# Calculate transition frequencies
transition_matrix = pd.crosstab(
    transitions_df['from_category'],
    transitions_df['to_category'],
    margins=False
)

# Calculate transition probabilities (normalize by row)
transition_probs = transition_matrix.div(transition_matrix.sum(axis=1), axis=0).fillna(0)

# Identify top transitions
all_transitions = []
for from_cat in transition_matrix.index:
    for to_cat in transition_matrix.columns:
        count = transition_matrix.loc[from_cat, to_cat]
        prob = transition_probs.loc[from_cat, to_cat]
        if count > 0:
            all_transitions.append({
                'from_category': from_cat,
                'to_category': to_cat,
                'count': count,
                'probability': prob,
                'same_category': from_cat == to_cat
            })

transitions_summary = pd.DataFrame(all_transitions).sort_values('count', ascending=False)


# Same-category retention
same_category = transitions_summary[transitions_summary['same_category'] == True]['count'].sum()
cross_category = transitions_summary[transitions_summary['same_category'] == False]['count'].sum()
total_trans = same_category + cross_category

print(f"\n Category Loyalty:")
print(f"Same category: {same_category:,} ({same_category/total_trans*100:.1f}%)")
print(f"Cross category: {cross_category:,} ({cross_category/total_trans*100:.1f}%)")

print(f"\n Top 10 Category Transitions:")
print("-" * 80)
for idx, row in transitions_summary.head(10).iterrows():
    transition_type = "SAME" if row['same_category'] else "CROSS"
    print(f"{row['from_category']:30s} to {row['to_category']:30s} | {row['count']:4d} ({row['probability']*100:5.1f}%) [{transition_type}]")

# Save transition data
transitions_summary.to_csv('../outputs/category_transitions.csv', index=False)

In [ ]:
# Focusing on 3 robust categories
focus_categories = ['watches_gifts', 'garden_tools', 'electronics']
category_names = {
    'watches_gifts': 'Watches/Gifts',
    'garden_tools': 'Garden Tools',
    'electronics': 'Electronics'
}

# Filter transitions involving focus categories
focus_transitions = transitions_df[
    (transitions_df['from_category'].isin(focus_categories)) |
    (transitions_df['to_category'].isin(focus_categories))
].copy()

print(f"Transitions involving focus categories: {len(focus_transitions):,}")

# Analyze outbound transitions from each focus category
for cat_key, cat_name in category_names.items():
    outbound = transitions_df[transitions_df['from_category'] == cat_key]
    
    if len(outbound) > 0:
        print(f"\n{cat_name} ({len(outbound)} transitions):")
        print("-" * 80)
        
        # Top destinations
        destinations = outbound['to_category'].value_counts().head(5)
        for dest_cat, count in destinations.items():
            pct = count / len(outbound) * 100
            dest_name = category_names.get(dest_cat, dest_cat)
            print(f"   to {dest_name:30s}: {count:4d} ({pct:5.1f}%)")
        
        # Average time to return
        avg_days = outbound['days_between'].mean()
        print(f"   Average days to next purchase: {avg_days:.1f}")
    else:
        print(f"\n{cat_name}: No transitions found")

In [ ]:
# Bucket level transitions
# Define bucket mapping
bucket_mapping = {
    'HOME_ESSENTIALS': ['bed_bath_table', 'housewares', 'furniture_decor', 'garden_tools', 
                        'home_appliances', 'kitchen_dining_laundry_garden_furniture', 
                        'home_construction', 'furniture_living_room', 'furniture_bedroom', 
                        'air_conditioning', 'home_confort', 'home_comfort_2', 
                        'furniture_mattress_and_upholstery'],
    'PERSONAL_CARE': ['health_beauty', 'perfumery', 'baby', 'diapers_and_hygiene'],
    'ELECTRONICS_TECH': ['computers_accessories', 'telephony', 'electronics', 'computers', 
                         'consoles_games', 'audio', 'tablets_printing_image', 'fixed_telephony',
                         'cine_photo'],
    'LEISURE_LIFESTYLE': ['sports_leisure', 'toys', 'watches_gifts', 'cool_stuff', 
                          'books_general_interest', 'musical_instruments', 'pet_shop', 
                          'fashion_bags_accessories', 'books_technical', 'books_imported',
                          'art', 'party_supplies', 'flowers', 'dvds_blu_ray', 'music',
                          'cds_dvds_musicals'],
    'AUTO_TOOLS': ['auto', 'construction_tools_construction', 'construction_tools_safety',
                   'costruction_tools_garden', 'construction_tools_lights', 
                   'costruction_tools_tools', 'signaling_and_security',
                   'agro_industry_and_commerce', 'industry_commerce_and_business'],
    'OFFICE_STATIONERY': ['stationery', 'office_furniture'],
    'FASHION_APPAREL': ['fashion_bags_accessories', 'fashion_shoes', 'fashion_male_clothing',
                        'fashion_underwear_beach', 'fashio_female_clothing', 'fashion_sport',
                        'fashion_childrens_clothes', 'luggage_accessories'],
    'FOOD_BEVERAGE': ['food', 'drinks', 'food_drink'],
    'SMALL_APPLIANCES': ['small_appliances', 'small_appliances_home_oven_and_coffee', 
                         'home_appliances_2']
}

# Create category to bucket mapping
cat_to_bucket = {}
for bucket, categories in bucket_mapping.items():
    for cat in categories:
        cat_to_bucket[cat] = bucket

# Map transitions to buckets
transitions_df['from_bucket'] = transitions_df['from_category'].map(cat_to_bucket)
transitions_df['to_bucket'] = transitions_df['to_category'].map(cat_to_bucket)

# Remove transitions with unmapped categories
bucket_transitions = transitions_df[
    transitions_df['from_bucket'].notna() & 
    transitions_df['to_bucket'].notna()
].copy()

# Calculate bucket transition matrix
bucket_trans_matrix = pd.crosstab(
    bucket_transitions['from_bucket'],
    bucket_transitions['to_bucket'],
    margins=False
)

# Calculate probabilities
bucket_trans_probs = bucket_trans_matrix.div(bucket_trans_matrix.sum(axis=1), axis=0).fillna(0)

# Bucket loyalty
bucket_transitions['same_bucket'] = bucket_transitions['from_bucket'] == bucket_transitions['to_bucket']
bucket_loyalty = bucket_transitions['same_bucket'].sum() / len(bucket_transitions) * 100

print(f"Bucket Transition Summary:")
print(f"Same bucket: {bucket_transitions['same_bucket'].sum():,} ({bucket_loyalty:.1f}%)")
print(f"Cross bucket: {(~bucket_transitions['same_bucket']).sum():,} ({100-bucket_loyalty:.1f}%)")

print(f"\n Top Cross-Bucket Transitions:")
print("-" * 80)

cross_bucket = bucket_transitions[~bucket_transitions['same_bucket']]
cross_bucket_summary = cross_bucket.groupby(['from_bucket', 'to_bucket']).size().reset_index(name='count')
cross_bucket_summary = cross_bucket_summary.sort_values('count', ascending=False)

for idx, row in cross_bucket_summary.head(10).iterrows():
    print(f"{row['from_bucket']:20s} to {row['to_bucket']:20s}: {row['count']:4d}")

In [ ]:
# Time to return analysis

# Overall distribution
print(f"Time Between Purchases:")
print(f"Mean: {transitions_df['days_between'].mean():.1f} days")
print(f"Median: {transitions_df['days_between'].median():.1f} days")
print(f"25th percentile: {transitions_df['days_between'].quantile(0.25):.1f} days")
print(f"75th percentile: {transitions_df['days_between'].quantile(0.75):.1f} days")

# By first category (focus categories)
print(f"\n Time-to-Return by First Purchase Category:")
print("-" * 80)

for cat_key, cat_name in category_names.items():
    cat_trans = transitions_df[transitions_df['from_category'] == cat_key]
    if len(cat_trans) > 0:
        print(f"   {cat_name:20s}: {cat_trans['days_between'].mean():6.1f} days (median: {cat_trans['days_between'].median():.1f})")


In [ ]:
# Recommendation framework
print("\n Recommendation Logic:")
print("-" * 80)

# For each focus category, identify top recommended next categories
for cat_key, cat_name in category_names.items():
    outbound = transitions_df[transitions_df['from_category'] == cat_key]
    
    if len(outbound) > 0:
        print(f"\n{cat_name} Buyers:")
        
        # Calculate probabilities
        next_cats = outbound['to_category'].value_counts()
        total = len(outbound)
        
        print("   Recommend (in order):")
        for rank, (next_cat, count) in enumerate(next_cats.head(3).items(), 1):
            prob = count / total * 100
            next_name = category_names.get(next_cat, next_cat)
            print(f"     {rank}. {next_name} ({prob:.1f}% probability)")


In [ ]:
# Visualizations:
# Time to return distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Overall distribution (capped at 365 days for clarity)
days_capped = transitions_df['days_between'].clip(upper=365)
ax1.hist(days_capped, bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
ax1.axvline(transitions_df['days_between'].median(), color='red', linestyle='--', 
            linewidth=2, label=f"Median: {transitions_df['days_between'].median():.0f} days")
ax1.axvline(transitions_df['days_between'].mean(), color='orange', linestyle='--',
            linewidth=2, label=f"Mean: {transitions_df['days_between'].mean():.0f} days")
ax1.set_xlabel('Days Between Purchases', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Time-to-Return Distribution\n(All Categories)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# By focus category
focus_cat_data = []
for cat_key, cat_name in category_names.items():
    cat_trans = transitions_df[transitions_df['from_category'] == cat_key]
    if len(cat_trans) > 0:
        focus_cat_data.append({
            'category': cat_name,
            'days': cat_trans['days_between'].values
        })

if focus_cat_data:
    positions = range(len(focus_cat_data))
    bp = ax2.boxplot([d['days'] for d in focus_cat_data], 
                      positions=positions,
                      labels=[d['category'] for d in focus_cat_data],
                      patch_artist=True,
                      widths=0.6)
    
    colors = ['#2E86AB', '#A23B72', '#F18F01']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax2.set_ylabel('Days Between Purchases', fontsize=12, fontweight='bold')
    ax2.set_title('Time-to-Return by Category\n(First Purchase)', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../outputs/images/time_to_return_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Category transition matrix
# Focus on top 10 categories by transaction volume
top_categories = transitions_df['from_category'].value_counts().head(10).index.tolist()

# Create subset transition matrix
subset_trans = transitions_df[
    transitions_df['from_category'].isin(top_categories) &
    transitions_df['to_category'].isin(top_categories)
]

trans_matrix_subset = pd.crosstab(
    subset_trans['from_category'],
    subset_trans['to_category'],
    margins=False
)

fig, ax = plt.subplots(figsize=(12, 10))

# Create heatmap
im = ax.imshow(trans_matrix_subset.values, cmap='YlOrRd', aspect='auto')

# Set ticks
ax.set_xticks(np.arange(len(trans_matrix_subset.columns)))
ax.set_yticks(np.arange(len(trans_matrix_subset.index)))
ax.set_xticklabels(trans_matrix_subset.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(trans_matrix_subset.index, fontsize=9)

ax.set_xlabel('Next Purchase Category', fontsize=12, fontweight='bold')
ax.set_ylabel('First Purchase Category', fontsize=12, fontweight='bold')
ax.set_title('Category Transition Matrix\n(Top 10 Categories by Volume)',
             fontsize=14, fontweight='bold', pad=20)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Number of Transitions', fontsize=11, fontweight='bold')

# Highlight diagonal (same-category repurchase)
for i in range(min(len(trans_matrix_subset.index), len(trans_matrix_subset.columns))):
    ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False, 
                               edgecolor='blue', linewidth=3))

plt.tight_layout()
plt.savefig('../outputs/images/category_transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Bucket loyalty
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Same bucket vs cross bucket
loyalty_data = [bucket_loyalty, 100-bucket_loyalty]
colors = ['#06A77D', '#D7263D']
labels = [f'Same Bucket\n{bucket_loyalty:.1f}%', f'Cross Bucket\n{100-bucket_loyalty:.1f}%']

wedges, texts, autotexts = ax1.pie(loyalty_data, labels=labels, colors=colors,
                                     autopct='', startangle=90,
                                     explode=(0.05, 0.05))

for text in texts:
    text.set_fontsize(12)
    text.set_fontweight('bold')

ax1.set_title('Bucket Loyalty in Sequential Purchases\n(Repeat Customers)',
              fontsize=14, fontweight='bold')

# Top cross-bucket flows
top_cross = cross_bucket_summary.head(8)
from_to_labels = [f"{row['from_bucket'][:12]}\n→ {row['to_bucket'][:12]}" 
                  for _, row in top_cross.iterrows()]

bars = ax2.barh(range(len(top_cross)), top_cross['count'].values,
                color='#F18F01', alpha=0.8, edgecolor='black', linewidth=1.5)

ax2.set_yticks(range(len(top_cross)))
ax2.set_yticklabels(from_to_labels, fontsize=9)
ax2.set_xlabel('Number of Transitions', fontsize=12, fontweight='bold')
ax2.set_title('Top Cross-Bucket Transitions\n(Exploration Opportunities)',
              fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/images/bucket_transition_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Recommendation framework
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (cat_key, cat_name) in enumerate(category_names.items()):
    ax = axes[idx]
    
    outbound = transitions_df[transitions_df['from_category'] == cat_key]
    
    if len(outbound) > 0:
        # Top 5 destinations
        next_cats = outbound['to_category'].value_counts().head(5)
        probs = (next_cats / len(outbound) * 100).values
        labels = [category_names.get(cat, cat[:20]) for cat in next_cats.index]
        
        bars = ax.barh(range(len(labels)), probs,
                      color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#95A3A6'][:len(labels)],
                      alpha=0.8, edgecolor='black', linewidth=1.5)
        
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=10)
        ax.set_xlabel('Probability (%)', fontsize=11, fontweight='bold')
        ax.set_title(f'After buying\n{cat_name}', fontsize=12, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        
        # Add value labels
        for i, (bar, prob) in enumerate(zip(bars, probs)):
            ax.text(prob + 1, i, f'{prob:.1f}%',
                   va='center', fontsize=10, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'No transitions\nfrom {cat_name}',
               ha='center', va='center', fontsize=12,
               transform=ax.transAxes)
        ax.axis('off')

plt.suptitle('Next Purchase Recommendations\n(Based on Historical Transition Probabilities)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/images/recommendation_framework.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Insights:
# 1. Repeat behavior: 
# 3% repeat rate and 97% first time buyers.
# Median return time is 29 days
# Mean is 79 days

# Category insight
# 46.9% stay in same category and 53.1% switch categories
# This is very different from the same order where majory of the orders stays in the same bundle. However in this analysis we discovered 53.1% switch categories
# Bucket loyalty is still strong
# 60.7% stay in same bucket
# 39.3% cross bucket purchase. There is an opportunity here.

# FOCUS CATEGORY INSIGHTS:
# Watches/Gifts (136 transitions):
# 52.2% repurchase same category (strong loyalty)
# 47.8% explore other categories
# Median return: 30 days (fast!)
# Top cross-category: housewares, fashion, bed/bath

# Garden Tools (103 transitions):
# 27.2% repurchase same (low loyalty)
# 72.8% explore (highest exploration rate)
# Median return: 66 days (slower)
# Top cross-category: furniture_decor (15.5%), housewares (7.8%)

# Electronics (42 transitions):
# 23.8% repurchase same (low loyalty)
# 76.2% explore
# Median return: 52 days
# Top cross-category: computers_accessories (9.5%), furniture, garden, watches

# Business implications
# Reason same-order bundle failed but sequential works:
# Same-order : Customers shop with specific intent. They have a need, they buy, and exit. No browsing behavior
# Sequential-order: 53.1% category switch. Customers explore overtime, Different shopping missions.
# Strategy: Focus on sequential not same order.

# Strategic recommendations:
# 1. Time based triggers: 
# Use 29 day median. 
# 2. Category specific recommendations:
# Watches/Gifts -> recommend housewares, fashion bags, bed/bath. 'Customers who bought watches also loved home essentials'
# Garden Tools -> furniture_decor, housewares. Complete your outdoor space with these complementary items
# Electronics -> computers_accessories, furniture, garden, watches. Tech buyers need these next
# 3. Bucket Bridge program:
# Current: 39.3% cross bucket naturally. Target: increase to 50% over 6 months
# Opportunities: Leisure -> Home, Home -> Personal_care, Electronice -> Home

# See paper Section 5.4 for complete category-level sequential purchase analysis
# and the full recommendation timing framework.
# SSRN: https://ssrn.com/abstract=6502262